# Read Data

## Notebook Plot Summary
This notebook loads processed 2p sessions, trializes the data, and generates several plot families.

- **YH24 timing histogram**: distribution of the offset between `vol_start` and the first `vol_stim_vis` pulse in each stimulus pair.
- **FOV / mask plots**: functional and anatomy images, ROI masks, and superimposed overlays.
- **Trialization outputs**: saves `neural_trials.h5` and `trial_labels.csv` after aligning voltage, Bpod, and imaging time.
- **Stimulus / reward / punish / lick aligned response plots**: pooled neural average traces and heatmaps across trial groups.
- **ISI distribution plot**: compares short- vs long-trial ISI distributions.
- **Clustering**: Gaussian mixture clustering of average activity between the first and second stimulus.
- **Decoding**: SVM-based decoding and temporal decoding analyses for trial categories.


## Output Layout

Notebook-generated files are now organized by mouse, session, and plot category under `Outputs/`.

- `Outputs/<mouse>/<session>/Figures/fov` stores field-of-view summaries
- `Outputs/<mouse>/<session>/Figures/stim` stores stimulus/ISI response plots
- `Outputs/<mouse>/<session>/Figures/clustering` stores clustering figures
- `Outputs/<mouse>/<session>/Figures/decoder` stores decoder figures and reports


* Reading directories

In [1]:
import os
import sys

# Set environment once and derive project/notebook roots from it.
ENVIRONMENT = 'LOCAL'  # 'LOCAL' or 'PACE'

if ENVIRONMENT == 'PACE':
    PROJECT_ROOT = '/storage/scratch1/3/grubin6/2AFC'
else:
    PROJECT_ROOT = os.path.join(os.path.expanduser('~'), '2AFC')

NOTEBOOK_DIR = os.path.join(PROJECT_ROOT, 'Test_pilot')

for path_entry in [PROJECT_ROOT, NOTEBOOK_DIR]:
    if path_entry not in sys.path:
        sys.path.insert(0, path_entry)

import numpy as np
import h5py
import matplotlib.pyplot as plt



In [2]:
from notebook_tools.io import (
    configure_trial_start_flags,
    resolve_session_path,
    get_session_output_dirs,
    read_ops,
    read_masks,
    visualize_calcium_signal,
    get_labeled_masks_img,
    create_superimposed_mask_images,
    plot_fov_summary,
    read_raw_voltages,
    read_dff,
    get_bpod_mat_path,
    read_bpod_mat_data,
    remove_start_impulse,
    correct_vol_start,
    get_trigger_time,
    infer_trial_starts_from_stim_pairs,
    get_trial_start_times,
    get_session_start_time,
    correct_time_img_center,
    save_trials,
    read_trial_label,
    read_neural_trials,
)

USE_VIS_STIM_PAIR_TRIAL_START_FALLBACK = True
FORCE_VIS_STIM_PAIR_TRIAL_START = False

# Use project-relative local paths to avoid repeating root directories.
local_mc11_home = os.path.join(os.path.expanduser('~'), 'MC11_2AFC')
local_mc11_project = os.path.join(PROJECT_ROOT, 'MC11_2AFC')
LOCAL_DATA_ROOT = local_mc11_home if os.path.exists(local_mc11_home) else local_mc11_project
PACE_CEDAR_DATA_ROOT = '/storage/cedar/cedar0/cedarp-fnajafi3-0/2p_imaging'
PACE_CEDAR_YH24_PROCESSED_ROOT = os.path.join(PACE_CEDAR_DATA_ROOT, 'processed', '2afc')
PACE_SCRATCH_MC11_ROOT = '/storage/scratch1/3/grubin6/2p_processing_results'
NOTEBOOK_OUTPUT_ROOT = os.path.join(NOTEBOOK_DIR, 'Outputs')

configure_trial_start_flags(USE_VIS_STIM_PAIR_TRIAL_START_FALLBACK, FORCE_VIS_STIM_PAIR_TRIAL_START)

YH24_DISTRIBUTION_SESSION_NAMES = [
    'YH24LG_CRBL_lobulev_20250701_2afc-533',
    'YH24LG_CRBL_simplex_20250528_2afc-373',
    'YH24LG_CRBL_simplex_20250529_2afc-379',
    'YH24LG_CRBL_simplex_20250530_2afc-389',
]

list_session_names = [
    'MC11_20260317_2afc_PFC-1326',
    'MC11_20260318_2afc_PFC-1329',
    'MC11_20260319_2afc_PFC-1332',
    'MC11_20260320_2afc_PFC-1335',
    'MC11_20260324_2afc_PFC-1342',
    'MC11_20260327_2afc_PFC-1359',
    'MC11_20260330_2afc_PFC-1377',
]

list_session_data_path = [
    resolve_session_path(
        session_name, ENVIRONMENT, LOCAL_DATA_ROOT, PACE_CEDAR_DATA_ROOT,
        PACE_CEDAR_YH24_PROCESSED_ROOT, PACE_SCRATCH_MC11_ROOT
    )
    for session_name in list_session_names
]

# Build ops objects once so downstream cells (read_masks/read_dff/read_raw_voltages) always have list_ops.
list_ops = read_ops(list_session_data_path)

list_session_output_path = get_session_output_dirs(list_session_names, NOTEBOOK_OUTPUT_ROOT)
primary_output_path = list_session_output_path[0] if len(list_session_output_path) > 0 else NOTEBOOK_OUTPUT_ROOT

FIGURE_OUTPUT_DIRS = {
    'fov': os.path.join(primary_output_path, 'Figures', 'fov'),
    'stim': os.path.join(primary_output_path, 'Figures', 'stim'),
    'clustering': os.path.join(primary_output_path, 'Figures', 'clustering'),
    'decoder': os.path.join(primary_output_path, 'Figures', 'decoder'),
}
for _dir in FIGURE_OUTPUT_DIRS.values():
    os.makedirs(_dir, exist_ok=True)

print('Session names:', list_session_names)
print('Session data path:')
for pth in list_session_data_path:
    print(' -', pth)
print('Loaded ops:', len(list_ops))




Session names: ['MC11_20260317_2afc_PFC-1326', 'MC11_20260318_2afc_PFC-1329', 'MC11_20260319_2afc_PFC-1332', 'MC11_20260320_2afc_PFC-1335', 'MC11_20260324_2afc_PFC-1342', 'MC11_20260327_2afc_PFC-1359', 'MC11_20260330_2afc_PFC-1377']
Session data path:
 - /Users/davisgrubin/MC11_2AFC/MC11_20260317_2afc_PFC-1326
 - /Users/davisgrubin/MC11_2AFC/MC11_20260318_2afc_PFC-1329
 - /Users/davisgrubin/MC11_2AFC/MC11_20260319_2afc_PFC-1332
 - /Users/davisgrubin/MC11_2AFC/MC11_20260320_2afc_PFC-1335
 - /Users/davisgrubin/MC11_2AFC/MC11_20260324_2afc_PFC-1342
 - /Users/davisgrubin/MC11_2AFC/MC11_20260327_2afc_PFC-1359
 - /Users/davisgrubin/MC11_2AFC/MC11_20260330_2afc_PFC-1377
Loaded ops: 7


In [ ]:
# from notebook_tools.distribution import _distribution_bpod_mat_path, _distribution_rising_edges, _distribution_first_vis_times
# print('Excluding YH24LG_CRBL_crux1_20250519_2afc-331 from this histogram because its Input0-to-first-Input1 offsets are a strong outlier.')
# MAX_HIST_DELAY_MS = 600
# yh24_offsets = []
# yh24_hist_offsets = []
# outlier_summary = []
# plt.figure(figsize=(8, 4))
# for session_name in YH24_DISTRIBUTION_SESSION_NAMES:
#     session_data_path = resolve_session_path(
#         session_name, ENVIRONMENT, LOCAL_DATA_ROOT, PACE_CEDAR_DATA_ROOT,
#         PACE_CEDAR_YH24_PROCESSED_ROOT, PACE_SCRATCH_MC11_ROOT)
#     stim_first, start_times = _distribution_first_vis_times(session_name, session_data_path)
#     offsets = stim_first - start_times
#     yh24_offsets.extend(offsets.tolist())
#     in_range_mask = offsets >= 0
#     outlier_offsets = offsets[~in_range_mask]
#     yh24_hist_offsets.extend(offsets[in_range_mask].tolist())
#     outlier_summary.append((session_name, int(np.sum(~in_range_mask)), outlier_offsets.tolist()))
#     print(f'{session_name}: n={len(offsets)}, mean={np.mean(offsets):.1f} ms, median={np.median(offsets):.1f} ms, >=0 ms={int(np.sum(in_range_mask))}, <0 ms={int(np.sum(~in_range_mask))}')
# plt.hist(yh24_hist_offsets, bins=40, color='steelblue', edgecolor='black')
# plt.xlabel('First vis_stim in pair - vol_start (ms)')
# plt.ylabel('Count')
# plt.title('YH24 distribution of trial-start to first-stimulus offsets')
# plt.tight_layout()
# print(f'Overall: n={len(yh24_offsets)}, mean={np.mean(yh24_offsets):.1f} ms, median={np.median(yh24_offsets):.1f} ms')
# outlier_total = sum(count for _, count, _ in outlier_summary)
# print(f'Excluded from histogram (<0ms): {outlier_total}')
# for session_name, count, outlier_offsets in outlier_summary:
#     if count > 0:
#         print(f'  {session_name}: {count} outliers, values={np.round(outlier_offsets, 1).tolist()}')



* Reading masks

In [ ]:
# Imported in setup cell above.


In [3]:
# label -1 --> excitattory neurons, 1 --> inhibitory neuons

[labels, masks, mean_func, max_func, mean_anat, masks_anat] = read_masks(list_ops[0])

## FOV Visualization
This section plots the functional and anatomy field-of-view images, colored calcium overlays, and ROI mask overlays. The summary figure is saved as `FOV_summary.pdf`.


In [4]:
# Build FOV plot inputs for all sessions so we can browse with a slider.
fov_mean_fun_channels = []
fov_max_fun_channels = []
fov_masks = []
fov_superimpose_func = []
fov_mean_anat = []
fov_superimpose_anat = []
fov_roi_counts = []

for ops in list_ops:
    labels_i, masks_i, mean_func_i, max_func_i, mean_anat_i, masks_anat_i = read_masks(ops)
    mean_fun_i, max_fun_i, super_func_i, super_anat_i = create_superimposed_mask_images(
        mean_func_i, max_func_i, masks_i, labels_i, mean_anat_i
    )
    fov_mean_fun_channels.append(mean_fun_i)
    fov_max_fun_channels.append(max_fun_i)
    fov_masks.append(masks_i)
    fov_superimpose_func.append(super_func_i)
    fov_mean_anat.append(mean_anat_i)
    fov_superimpose_anat.append(super_anat_i)
    fov_roi_counts.append(len(labels_i))

print(f'Prepared FOV data for {len(fov_mean_fun_channels)} sessions.')




Prepared FOV data for 7 sessions.


In [ ]:
# Imported in setup cell above.


In [ ]:
# Static single-session FOV plot disabled to avoid duplicate outputs.
# Use the interactive arrows cell below instead.


In [5]:
# Interactive per-session FOV viewer (one figure at a time)
plot_fov_summary(
    fov_mean_fun_channels,
    fov_max_fun_channels,
    fov_masks,
    fov_superimpose_func,
    fov_mean_anat,
    fov_superimpose_anat,
    session_names=list_session_names,
    use_slider=True,
    navigator='arrows',
    roi_counts=fov_roi_counts
)




HTML(value='')

Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x10\x9c\x00\x00\x0b\xbf\x08\x06\x00\x00\x00\xa7q%\xe…

* Read Raw voltage

In [ ]:
# Imported in setup cell above.


In [6]:
# raw_voltages.h5 may be unavailable when downloaded with --exclude raw_voltages.h5.
raw_voltages_path = os.path.join(list_ops[0]['save_path0'], 'raw_voltages.h5')
if os.path.exists(raw_voltages_path):
    [vol_time, vol_start, vol_stim_vis, vol_img,
                vol_hifi, vol_stim_aud, vol_flir,
                vol_pmt, vol_led] = read_raw_voltages(list_ops[0])
    print(f'Loaded raw voltages: {raw_voltages_path}')
else:
    vol_time = vol_start = vol_stim_vis = vol_img = None
    vol_hifi = vol_stim_aud = vol_flir = vol_pmt = vol_led = None
    print(f'Skipping raw voltage load (missing): {raw_voltages_path}')



Skipping raw voltage load (missing): /Users/davisgrubin/MC11_2AFC/MC11_20260317_2afc_PFC-1326/raw_voltages.h5


* Read DFF

In [ ]:
# Imported in setup cell above.


In [ ]:
dff = read_dff(list_ops[0])

### Raw DFF Trace Quality Viewer
Interactively inspect raw DFF traces by session and neuron.


In [7]:
import importlib, notebook_tools.plot_viewers as plot_viewers
importlib.reload(plot_viewers)
from notebook_tools.plot_viewers import show_raw_dff_trace_viewer

show_raw_dff_trace_viewer(
    list_session_names,
    list_ops,
    read_dff,
    max_points=10000,
    max_all_neurons=80,
)



### Click FOV ROI to Inspect DFF
Click an ROI centroid on the FOV to display that neuron trace below.


In [31]:
# %matplotlib widget
import importlib
import notebook_tools.plot_viewers as plot_viewers
importlib.reload(plot_viewers)
plot_viewers.show_fov_dff_click_viewer(
    list_session_names,
    list_ops,
    read_masks,
    read_dff,
    max_points=10000,
)



* Read bpod

In [ ]:
# Imported in setup cell above.


In [11]:
# I/O helpers are imported in the setup cell; this optional reload keeps notebook_tools.io edits in sync.
import importlib
import notebook_tools.io as nb_io
importlib.reload(nb_io)


<module 'notebook_tools.io' from '/Users/davisgrubin/2AFC/notebook_tools/io.py'>

### Trialization Save Step
This cell loads dF/F and raw voltages, aligns Bpod timing to the DAQ/imaging clock, and saves `neural_trials.h5` plus `trial_labels.csv` for downstream analyses.


In [12]:
if globals().get('vol_time') is None:
    print('Skipping trialization from raw voltages because raw_voltages.h5 is missing or has not been loaded.')
    print('Using existing neural_trials.h5 and trial_labels.csv from session folder instead.')
else:
    print('Reading dff traces and voltage recordings')
    vol_stim_vis = remove_start_impulse(vol_time, vol_stim_vis)
    vol_stim_vis = correct_vol_start(vol_stim_vis)
    session_start_time = get_session_start_time(vol_time, vol_start, list_ops[0], vol_stim_vis)
    trial_labels = read_bpod_mat_data(list_ops[0], session_start_time)
    print('Correcting 2p camera trigger time')
    time_img, _ = get_trigger_time(vol_time, vol_img)
    time_neuro = correct_time_img_center(time_img)
    print('Saving trial data')
    save_trials(
        list_ops[0], time_neuro, dff, trial_labels,
        vol_time, vol_stim_vis,
        vol_stim_aud, vol_flir,
        vol_pmt, vol_led)



Skipping trialization from raw voltages because raw_voltages.h5 is missing or has not been loaded.
Using existing neural_trials.h5 and trial_labels.csv from session folder instead.


* Reading Trialization

In [13]:
# Imported in setup cell above.


In [14]:
neural_trials = read_neural_trials(list_ops[0], 1)
trial_labels = read_trial_label(list_ops[0])

* Align around stim/ reward/ punish/ first lick

In [15]:
from notebook_tools.alignment import trim_seq, get_lick_response, get_perception_response


# Analysis figure

In [16]:
from notebook_tools.lick import get_roi_label_color, apply_colormap, adjust_layout_heatmap, plot_gridspec_subplot, plot_gridspec_superimposed, plot_gridspec_heatmap, plot_sorted_heatmap, calculate_metrics, set_uniform_ylim, main


* Stim 

In [17]:
from notebook_tools.stim import get_roi_label_color, apply_colormap, adjust_layout_heatmap, plot_gridspec_subplot, plot_gridspec_superimposed, plot_gridspec_heatmap, plot_sorted_heatmap, compute_trial_masks, calculate_metrics, set_uniform_ylim, pool_session_data, main


In [18]:
import importlib
import notebook_tools.alignment as nb_alignment
import notebook_tools.stim as nb_stim
importlib.reload(nb_alignment)
importlib.reload(nb_stim)



<module 'notebook_tools.stim' from '/Users/davisgrubin/2AFC/notebook_tools/stim.py'>

### ISI Distribution Plot
This cell builds short- and long-trial ISI distributions from the trialized session data and plots their idealized distributions for quick sanity checks.


In [19]:
from notebook_tools.plot_viewers import run_isi_distribution_for_sessions

isi_distribution_summaries, isi_distribution_summary_df = run_isi_distribution_for_sessions(
    list_session_names,
    list_ops,
    list_session_output_path,
    read_trial_label,
)
display(isi_distribution_summary_df)




--- ISI distribution MC11_20260317_2afc_PFC-1326 ---
{'session': 'MC11_20260317_2afc_PFC-1326', 'status': 'ok', 'n_short': 163, 'n_long': 152, 'short_mean': 779.768720381099, 'long_mean': 2239.557208412572, 'short_min': 501.70001220703125, 'short_max': 1101.9000244140625, 'long_min': 1901.699951171875, 'long_max': 2502.0, 'png_path': '/Users/davisgrubin/2AFC/Test_pilot/Outputs/MC11/MC11_20260317_2afc_PFC-1326/Figures/stim/isi_distribution.png', 'pdf_path': '/Users/davisgrubin/2AFC/Test_pilot/Outputs/MC11/MC11_20260317_2afc_PFC-1326/Figures/stim/isi_distribution.pdf', 'error': None}

--- ISI distribution MC11_20260318_2afc_PFC-1329 ---
{'session': 'MC11_20260318_2afc_PFC-1329', 'status': 'ok', 'n_short': 193, 'n_long': 203, 'short_mean': 814.4507873218912, 'long_mean': 2193.7861676897323, 'short_min': 501.79998779296875, 'short_max': 1102.0, 'long_min': 1901.800048828125, 'long_max': 2502.0, 'png_path': '/Users/davisgrubin/2AFC/Test_pilot/Outputs/MC11/MC11_20260318_2afc_PFC-1329/Figure

,session,status,n_short,n_long,short_mean,long_mean,short_min,short_max,long_min,long_max,png_path,pdf_path,error
0,MC11_20260317_2afc_PFC-1326,ok,163,152,779.768720,2239.557208,501.700012,1101.900024,1901.699951,2502.000000,/Users/davisgrubin/2AFC/Test_pilot/Outputs/MC1...,/Users/davisgrubin/2AFC/Test_pilot/Outputs/MC1...,None
1,MC11_20260318_2afc_PFC-1329,ok,193,203,814.450787,2193.786168,501.799988,1102.000000,1901.800049,2502.000000,/Users/davisgrubin/2AFC/Test_pilot/Outputs/MC1...,/Users/davisgrubin/2AFC/Test_pilot/Outputs/MC1...,None
2,MC11_20260319_2afc_PFC-1332,ok,176,182,795.992051,2202.955468,501.899994,1102.099976,1901.800049,2535.300049,/Users/davisgrubin/2AFC/Test_pilot/Outputs/MC1...,/Users/davisgrubin/2AFC/Test_pilot/Outputs/MC1...,None
3,MC11_20260320_2afc_PFC-1335,ok,188,182,832.447350,2174.724145,501.899994,1102.099976,1901.800049,2502.000000,/Users/davisgrubin/2AFC/Test_pilot/Outputs/MC1...,/Users/davisgrubin/2AFC/Test_pilot/Outputs/MC1...,None
4,MC11_20260324_2afc_PFC-1342,ok,124,113,801.921784,2190.125629,501.799988,1102.000000,1901.800049,2502.000000,/Users/davisgrubin/2AFC/Test_pilot/Outputs/MC1...,/Users/davisgrubin/2AFC/Test_pilot/Outputs/MC1...,None
5,MC11_20260327_2afc_PFC-1359,ok,222,226,NaN,2188.708837,NaN,NaN,1901.900024,2502.100098,/Users/davisgrubin/2AFC/Test_pilot/Outputs/MC1...,/Users/davisgrubin/2AFC/Test_pilot/Outputs/MC1...,None
6,MC11_20260330_2afc_PFC-1377,ok,207,198,807.016433,2193.741393,501.799988,1102.099976,1901.800049,2502.100098,/Users/davisgrubin/2AFC/Test_pilot/Outputs/MC1...,/Users/davisgrubin/2AFC/Test_pilot/Outputs/MC1...,None


In [20]:
from notebook_tools.plot_viewers import show_isi_distribution_viewer

show_isi_distribution_viewer(isi_distribution_summaries)



HTML(value='')

HTML(value='')

Image(value=b'')

# Clustering


### Clustering Analysis
This section averages neural activity between the first and second stimulus, clusters neurons with GMM-based methods, and optionally saves clustering outputs if a path is provided.


In [21]:
from notebook_tools.clustering import find_min_long_trial_isi


In [22]:
l_frames = 0
_, r_frames = find_min_long_trial_isi(neural_trials)
neu_seq, neu_time, stim_seq, stim_value, stim_time, led_value, trial_type, block_type, isi, decision, outcome = get_perception_response(
      neural_trials, 'stim_seq', l_frames, r_frames, indices=0
  )

100%|██████████| 315/315 [00:00<00:00, 9638.63it/s]


In [23]:
long_trials = trial_type == 1
avg_neuron = np.nanmean(neu_seq[long_trials], axis = 0)
avg_neuron.shape

(19, 63)

In [24]:
# PROJECT_ROOT and NOTEBOOK_DIR are already added to sys.path in setup.


In [25]:
from notebook_tools.clustering import (
    NeuralActivityClustering,
    main,
    pool_session_data,
    find_min_long_trial_isi,
    get_avg_neural_activity,
    clustering_GMM,
)
import pandas as pd

clustering_summaries = []

for session_name, ops, output_path in zip(list_session_names, list_ops, list_session_output_path):
    print(f"\n--- Clustering {session_name} ---")
    labels_i = None
    trial_labels_i = None
    try:
        labels_i, *_ = read_masks(ops)
        neural_trials_i = read_neural_trials(ops, 1)
        trial_labels_i = read_trial_label(ops)
        result = clustering_GMM(
            neural_trials_i,
            labels_i,
            save_path=output_path,
            pooling=False,
            show_plots=False,
        )
        summary = {
            'session': session_name,
            'status': 'ok',
            'n_neurons': int(len(labels_i)),
            'n_trials': int(len(trial_labels_i)),
            'optimal_k': int(result['optimal_k']),
            'cluster_counts': result['cluster_counts'],
            'silhouette': float(result['final_metrics']['silhouette']),
            'bic': float(result['final_metrics']['bic']),
            'aic': float(result['final_metrics']['aic']),
            'final_reg_covar': result.get('final_reg_covar'),
            'error': None,
        }
        print(summary)
    except ValueError as exc:
        summary = {
            'session': session_name,
            'status': 'skipped',
            'n_neurons': int(len(labels_i)) if labels_i is not None else None,
            'n_trials': int(len(trial_labels_i)) if trial_labels_i is not None else None,
            'error': f'{type(exc).__name__}: {exc}',
        }
        print(f"Skipping {session_name}: {summary['error']}")
    except Exception as exc:
        summary = {
            'session': session_name,
            'status': 'failed',
            'n_neurons': int(len(labels_i)) if labels_i is not None else None,
            'n_trials': int(len(trial_labels_i)) if trial_labels_i is not None else None,
            'error': f'{type(exc).__name__}: {exc}',
        }
        print(f"Failed {session_name}: {summary['error']}")
    clustering_summaries.append(summary)

clustering_summary_df = pd.DataFrame(clustering_summaries)
display(clustering_summary_df)





--- Clustering MC11_20260317_2afc_PFC-1326 ---


100%|██████████| 315/315 [00:00<00:00, 10861.70it/s]


Preprocessing neural activity data...
PCA applied: 9 components explain 0.952 of variance
Finding optimal number of clusters (max=15)...
Optimal number of clusters: 2
Fitting final GMM model with 2 clusters...
Final clustering metrics:
  bic: 764.698
  aic: 729.754
  silhouette: 0.346
  calinski_harabasz: 11.161
  log_likelihood: -17.257
{'session': 'MC11_20260317_2afc_PFC-1326', 'status': 'ok', 'n_neurons': 19, 'n_trials': 315, 'optimal_k': 2, 'cluster_counts': [13, 6], 'silhouette': 0.34603139758110046, 'bic': 764.6981226626482, 'aic': 729.7538804334899, 'final_reg_covar': 1e-06, 'error': None}

--- Clustering MC11_20260318_2afc_PFC-1329 ---


100%|██████████| 396/396 [00:00<00:00, 9775.90it/s]


Preprocessing neural activity data...
PCA applied: 13 components explain 0.962 of variance
Finding optimal number of clusters (max=15)...
Optimal number of clusters: 2
Fitting final GMM model with 2 clusters...
Final clustering metrics:
  bic: 1244.675
  aic: 1184.494
  silhouette: 0.251
  calinski_harabasz: 8.281
  log_likelihood: -23.446
{'session': 'MC11_20260318_2afc_PFC-1329', 'status': 'ok', 'n_neurons': 133, 'n_trials': 396, 'optimal_k': 2, 'cluster_counts': [14, 9], 'silhouette': 0.25144824385643005, 'bic': 1244.6748947384729, 'aic': 1184.493701294228, 'final_reg_covar': 1e-06, 'error': None}

--- Clustering MC11_20260319_2afc_PFC-1332 ---


...

100%|██████████| 448/448 [00:00<00:00, 9541.08it/s]


Preprocessing neural activity data...
PCA applied: 19 components explain 0.952 of variance
Finding optimal number of clusters (max=15)...
Optimal number of clusters: 2
Fitting final GMM model with 2 clusters...
Final clustering metrics:
  bic: 8598.549
  aic: 8377.158
  silhouette: 0.105
  calinski_harabasz: 21.112
  log_likelihood: -31.386
{'session': 'MC11_20260327_2afc_PFC-1359', 'status': 'ok', 'n_neurons': 131, 'n_trials': 448, 'optimal_k': 2, 'cluster_counts': [104, 27], 'silhouette': 0.10505863279104233, 'bic': 8598.548532373647, 'aic': 8377.158338487157, 'final_reg_covar': 1e-06, 'error': None}

--- Clustering MC11_20260330_2afc_PFC-1377 ---


100%|██████████| 405/405 [00:00<00:00, 9988.73it/s]


Preprocessing neural activity data...
PCA applied: 20 components explain 0.954 of variance
Finding optimal number of clusters (max=15)...
Optimal number of clusters: 2
Fitting final GMM model with 2 clusters...
Final clustering metrics:
  bic: 15381.090
  aic: 15105.474
  silhouette: 0.182
  calinski_harabasz: 58.281
  log_likelihood: -33.656
{'session': 'MC11_20260330_2afc_PFC-1377', 'status': 'ok', 'n_neurons': 222, 'n_trials': 405, 'optimal_k': 2, 'cluster_counts': [107, 115], 'silhouette': 0.1822848618030548, 'bic': 15381.090493765421, 'aic': 15105.473625833767, 'final_reg_covar': 1e-06, 'error': None}


,session,status,n_neurons,n_trials,optimal_k,cluster_counts,silhouette,bic,aic,final_reg_covar,error
0,MC11_20260317_2afc_PFC-1326,ok,19,315,2,"[13, 6]",0.346031,764.698123,729.753880,0.000001,None
1,MC11_20260318_2afc_PFC-1329,ok,133,396,2,"[14, 9]",0.251448,1244.674895,1184.493701,0.000001,None
2,MC11_20260319_2afc_PFC-1332,ok,164,358,2,"[120, 44]",0.162251,10598.860200,10360.170485,0.000001,None
3,MC11_20260320_2afc_PFC-1335,ok,123,370,5,"[10, 9, 22, 67, 15]",0.010518,9682.331078,9024.279939,0.000001,None
4,MC11_20260324_2afc_PFC-1342,ok,68,237,2,"[54, 14]",0.154277,4624.774674,4462.750612,0.000001,None
5,MC11_20260327_2afc_PFC-1359,ok,131,448,2,"[104, 27]",0.105059,8598.548532,8377.158338,0.000001,None
6,MC11_20260330_2afc_PFC-1377,ok,222,405,2,"[107, 115]",0.182285,15381.090494,15105.473626,0.000001,None


In [26]:
from notebook_tools.plot_viewers import show_clustering_plot_viewer

show_clustering_plot_viewer(
    clustering_summaries,
    list_session_names,
    list_session_output_path,
)



HTML(value='')

HTML(value='')

Image(value=b'')

# Decoding

Q: during ITI (before stim onset), is there info in the pop about block type? 
- use majority trials;
- - align on stim 1: timepoints before stim1 (ITI), and right after stim 1 
- - align on stim 2: timepoints before stim 2 (ISI), and right after stim 2 

### Decoding Analysis
This section runs SVM decoding, shuffle controls, and temporal decoding plots for trial-label categories, saving figures under the chosen session output path.


In [27]:
from notebook_tools.decoding import (
    neural_decoder_svm,
    pool_session_data,
    temporal_decoding_accuracy,
    plot_temporal_decoding_accuracy,
    plot_shuffle_vs_normal,
    generate_decoder_labels,
    generate_majority_labels,
    generate_majority_rare_labels,
    separate_rare_trials,
    decoder_decision,
)
import pandas as pd

DECODING_MODE = 'rare vs common'
DECODING_INDICE = 2
DECODING_L_FRAMES = 90
DECODING_R_FRAMES = 120

decoding_summaries = []

for session_name, ops, output_path in zip(list_session_names, list_ops, list_session_output_path):
    print(f"\n--- Decoding {session_name} ---")
    labels_i = None
    trial_labels_i = None
    try:
        labels_i, *_ = read_masks(ops)
        neural_trials_i = read_neural_trials(ops, 1)
        trial_labels_i = read_trial_label(ops)
        result = decoder_decision(
            neural_trials_i,
            labels_i,
            DECODING_MODE,
            DECODING_INDICE,
            l_frames=DECODING_L_FRAMES,
            r_frames=DECODING_R_FRAMES,
            save_path=output_path,
            pooling=False,
        )
        summary = {
            'session': session_name,
            'status': 'ok',
            'n_neurons': int(len(labels_i)),
            'n_trials': int(len(trial_labels_i)),
            'best_cv_accuracy': result.get('best_cv_accuracy'),
            'test_balanced_accuracy': result.get('test_balanced_accuracy'),
            'peak_temporal_accuracy': result.get('peak_temporal_accuracy'),
            'error': None,
        }
        print(summary)
    except ValueError as exc:
        summary = {
            'session': session_name,
            'status': 'skipped',
            'n_neurons': int(len(labels_i)) if labels_i is not None else None,
            'n_trials': int(len(trial_labels_i)) if trial_labels_i is not None else None,
            'error': f'{type(exc).__name__}: {exc}',
        }
        print(f"Skipping {session_name}: {summary['error']}")
    except Exception as exc:
        summary = {
            'session': session_name,
            'status': 'failed',
            'n_neurons': int(len(labels_i)) if labels_i is not None else None,
            'n_trials': int(len(trial_labels_i)) if trial_labels_i is not None else None,
            'error': f'{type(exc).__name__}: {exc}',
        }
        print(f"Failed {session_name}: {summary['error']}")
    decoding_summaries.append(summary)

decoding_summary_df = pd.DataFrame(decoding_summaries)
display(decoding_summary_df)




--- Decoding MC11_20260317_2afc_PFC-1326 ---


100%|██████████| 315/315 [00:00<00:00, 9567.51it/s]


=== Neural Decoder Analysis ===
Original data shape: (311, 19, 210)
Using unbinned data
Flattened feature matrix shape: (0, 3990)

=== Label Distribution ===
Failed MC11_20260317_2afc_PFC-1326: ZeroDivisionError: division by zero

--- Decoding MC11_20260318_2afc_PFC-1329 ---


100%|██████████| 396/396 [00:00<00:00, 8817.64it/s]


=== Neural Decoder Analysis ===
Original data shape: (392, 23, 210)
Using unbinned data
Flattened feature matrix shape: (0, 4830)

=== Label Distribution ===
Failed MC11_20260318_2afc_PFC-1329: ZeroDivisionError: division by zero

--- Decoding MC11_20260319_2afc_PFC-1332 ---


...

100%|██████████| 448/448 [00:00<00:00, 8665.04it/s]


=== Neural Decoder Analysis ===
Original data shape: (444, 131, 210)
Using unbinned data
Flattened feature matrix shape: (32, 27510)

=== Label Distribution ===
Long_common: 18 trials (56.2%)
Long_rare: 14 trials (43.8%)

=== Undersampling Majority Classes ===
Undersampling to 14 trials per class
New label distribution:
Long_common: 14 trials (50.0%)
Long_rare: 14 trials (50.0%)

=== Train-Test Split (80/20) ===
Training set: 22 trials
Test set: 6 trials

=== Feature Scaling ===
Applied z-score normalization

=== SVM Training (C=0.1) ===

=== 5-Fold Cross-Validation ===
CV Balanced Accuracy: 0.500 ± 0.000
CV Scores: ['0.500', '0.500', '0.500', '0.500', '0.500']

=== Final Results ===
Training Balanced Accuracy: 1.000
Test Balanced Accuracy: 0.667

=== Classification Report (Test Set) ===
               precision    recall  f1-score   support

0_Long_common       1.00      0.33      0.50         3
  1_Long_rare       0.60      1.00      0.75         3

     accuracy                     

100%|██████████| 405/405 [00:00<00:00, 8936.87it/s]


=== Neural Decoder Analysis ===
Original data shape: (401, 222, 210)
Using unbinned data
Flattened feature matrix shape: (60, 46620)

=== Label Distribution ===
Long_common: 44 trials (73.3%)
Long_rare: 16 trials (26.7%)

=== Undersampling Majority Classes ===
Undersampling to 16 trials per class
New label distribution:
Long_common: 16 trials (50.0%)
Long_rare: 16 trials (50.0%)

=== Train-Test Split (80/20) ===
Training set: 25 trials
Test set: 7 trials

=== Feature Scaling ===
Applied z-score normalization

=== SVM Training (C=0.1) ===

=== 5-Fold Cross-Validation ===
CV Balanced Accuracy: 0.433 ± 0.143
CV Scores: ['0.500', '0.500', '0.167', '0.583', '0.417']

=== Final Results ===
Training Balanced Accuracy: 1.000
Test Balanced Accuracy: 0.750

=== Classification Report (Test Set) ===
               precision    recall  f1-score   support

0_Long_common       1.00      0.50      0.67         4
  1_Long_rare       0.60      1.00      0.75         3

     accuracy                     

,session,status,n_neurons,n_trials,error,best_cv_accuracy,test_balanced_accuracy,peak_temporal_accuracy
0,MC11_20260317_2afc_PFC-1326,failed,19,315,ZeroDivisionError: division by zero,NaN,NaN,NaN
1,MC11_20260318_2afc_PFC-1329,failed,133,396,ZeroDivisionError: division by zero,NaN,NaN,NaN
2,MC11_20260319_2afc_PFC-1332,ok,164,358,None,0.500000,0.666667,0.675000
3,MC11_20260320_2afc_PFC-1335,ok,123,370,None,0.500000,0.500000,0.775000
4,MC11_20260324_2afc_PFC-1342,ok,68,237,None,0.483333,0.500000,0.733333
5,MC11_20260327_2afc_PFC-1359,ok,131,448,None,0.500000,0.666667,0.816667
6,MC11_20260330_2afc_PFC-1377,ok,222,405,None,0.433333,0.750000,0.750000


In [28]:
from notebook_tools.plot_viewers import show_decoding_plot_viewer

show_decoding_plot_viewer(
    decoding_summaries,
    list_session_names,
    list_session_output_path,
    DECODING_MODE,
)



HTML(value='')

HTML(value='')

Image(value=b'')

### Interactive Session Plot Comparison
Use dropdowns to choose two sessions and compare the same generated plot side by side.


In [29]:
from notebook_tools.plot_viewers import show_session_plot_comparison

show_session_plot_comparison(list_session_names, list_session_output_path)

